# SOXLBacktest Teacher Risk-Budget Feasibility V1

Purpose: test whether the qlib-aligned SOXLBacktest buy/sell path can be converted into a non-leaking teacher target that preserves enough `wf_2020` bull participation while satisfying the walk-forward MaxDD budget.

This is a diagnostic target-recipe screen, not actor retention. It is intentionally placed before another training run because the latest checkpoint metric replay still fails `wf_2020` after metric-source reconciliation.

Governance:
- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1`.
- Direct comparison baselines: `soxlbacktest_qlib_walkforward_baseline_v1`, `cap10_three_fold_checkpoint_metric_replay_v2`, `cap10_training_objective_handoff_v1`.
- Role: `diagnose`.
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024`.
- Gate: relative return >= +30pp and MaxDD < 20% for every required fold.
- Changed layer: model_selection / training_target / execution_budget_feasibility.
- Hypothesis: a SOXLBacktest teacher path plus non-forward-looking risk caps might be sufficient to define the next retained training target without another blind PPO run.
- Failure mode: caps that protect `wf_2022` also destroy `wf_2020` bull participation, proving that teacher cap overlays are not enough.


In [1]:

from __future__ import annotations

import itertools
import json
import math
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

EXPERIMENT = 'soxlbacktest_teacher_risk_budget_feasibility_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

SOURCE_BASELINE_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'soxlbacktest_qlib_walkforward_baseline_v1'
SOURCE_REPLAY_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_three_fold_checkpoint_metric_replay_v2'
SOURCE_HANDOFF_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_training_objective_handoff_v1'
SOURCE_DAILY = SOURCE_BASELINE_CARD / 'candidate_daily.csv'
SOURCE_METRICS = SOURCE_BASELINE_CARD / 'candidate_metrics.csv'
SOURCE_REPLAY_SELECTION = SOURCE_REPLAY_CARD / 'selection.json'
SOURCE_REPLAY_FOLD = SOURCE_REPLAY_CARD / 'checkpoint_selection_by_fold.csv'
SOURCE_HANDOFF_SELECTION = SOURCE_HANDOFF_CARD / 'selection.json'

TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 20.0
WINDOWS = {
    'wf_2020': ('2020-01-01', '2020-12-31'),
    'wf_2022': ('2022-01-01', '2022-12-31'),
    'wf_2023_2024': ('2024-01-01', '2024-12-31'),
}


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value.relative_to(ROOT)) if value.is_absolute() and ROOT in value.parents else str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')

print('card', CARD.relative_to(ROOT))


card runtime/model_selection/card/soxlbacktest_teacher_risk_budget_feasibility_v1


In [2]:

panel = pd.read_csv(ROOT / 'data' / 'processed' / 'panels' / 'soxl_soxx_vix_full.csv', parse_dates=['date']).sort_values('date').set_index('date')
price = panel['soxl_close'].astype(float)
legacy_daily = pd.read_csv(SOURCE_DAILY, parse_dates=['date'])
legacy_metrics = pd.read_csv(SOURCE_METRICS)
legacy_position = legacy_daily[legacy_daily['candidate'].eq('legacy_default')].set_index('date')['position'].astype(float)
if legacy_position.max() > 1.5:
    legacy_position = legacy_position / 100.0
legacy_position = legacy_position.reindex(panel.index).ffill().fillna(0.0).clip(0.0, 1.0)

replay_selection = json.loads(SOURCE_REPLAY_SELECTION.read_text(encoding='utf-8'))
replay_by_fold = pd.read_csv(SOURCE_REPLAY_FOLD)
handoff_selection = json.loads(SOURCE_HANDOFF_SELECTION.read_text(encoding='utf-8'))

# Lagged, non-forward-looking features for teacher-cap diagnostics.
ret = price.pct_change().fillna(0.0)
lag_price = price.shift(1)
ma63 = price.rolling(63, min_periods=30).mean().shift(1)
ma180 = price.rolling(180, min_periods=80).mean().shift(1)
feature_frame = pd.DataFrame(index=panel.index)
feature_frame['vixq_252'] = panel['vix'].astype(float).shift(1).rolling(252, min_periods=80).rank(pct=True)
feature_frame['realvol_21'] = ret.rolling(21, min_periods=10).std().shift(1) * np.sqrt(252)
feature_frame['trend63'] = lag_price / ma63 - 1.0
feature_frame['trend180'] = lag_price / ma180 - 1.0
feature_frame['ret21'] = price.pct_change(21).shift(1)
feature_frame['ret63'] = price.pct_change(63).shift(1)
feature_frame['dd63'] = lag_price / lag_price.rolling(63, min_periods=20).max() - 1.0
feature_frame = feature_frame.replace([np.inf, -np.inf], np.nan)

historical_context = {
    'experiment': EXPERIMENT,
    'closest_success_case': {
        'card': 'soxlbacktest_qlib_walkforward_baseline_v1',
        'why': 'SOXLBacktest legacy_default has sufficient wf_2020/YTD raw return but violates walk-forward drawdown budgets.',
    },
    'closest_failure_case': {
        'card': 'cap10_three_fold_checkpoint_metric_replay_v2',
        'why': 'Corrected checkpoint replay passes wf_2022 and wf_2023_2024 but fails wf_2020 after same-source buyhold metric alignment.',
    },
    'why_not_duplicate': 'This screen tests SOXLBacktest teacher position paths plus explicit risk-budget caps before training; it is not another BC/PPO checkpoint rerun and does not use OOS labels as supervised training targets.',
    'source_replay_decision': replay_selection.get('decision'),
    'source_replay_failed_folds': replay_selection.get('failed_folds'),
    'source_handoff_decision': handoff_selection.get('decision'),
    'source_handoff_objective_terms': handoff_selection.get('objective_terms'),
}
write_json(CARD / 'historical_context.json', historical_context)

context_rows = []
for _, row in legacy_metrics[legacy_metrics['candidate'].isin(['legacy_default', 'wf_pass_ytd18_15_15_02', 'low_risk_15_05_02'])].iterrows():
    if row['window'] not in WINDOWS:
        continue
    context_rows.append({
        'source': 'soxlbacktest_qlib_walkforward_baseline_v1',
        'candidate': row['candidate'],
        'window': row['window'],
        'return': float(row['rulebase_return_pct']),
        'buyhold_return': float(row['soxl_buyhold_return_pct']),
        'relative_return': float(row['rulebase_return_pct'] - row['soxl_buyhold_return_pct']),
        'max_dd': float(row['rulebase_max_drawdown_pct']),
        'avg_pos': float(row['average_position_pct']) / 100.0,
        'gate': bool((row['rulebase_return_pct'] - row['soxl_buyhold_return_pct']) >= TARGET_RELATIVE_RETURN and row['rulebase_max_drawdown_pct'] < TARGET_MAX_DD),
    })
for _, row in replay_by_fold.iterrows():
    context_rows.append({
        'source': 'cap10_three_fold_checkpoint_metric_replay_v2',
        'candidate': row['selected_candidate'],
        'window': row['fold'],
        'return': None,
        'buyhold_return': None,
        'relative_return': float(row['selected_relative_return']),
        'max_dd': float(row['selected_max_dd']),
        'avg_pos': None,
        'gate': bool(row['selected_candidate_gate']),
    })
context = pd.DataFrame(context_rows)
context.to_csv(CARD / 'historical_gate_context.csv', index=False)
display(context)


,source,candidate,window,return,buyhold_return,relative_return,max_dd,avg_pos,gate
0,soxlbacktest_qlib_walkforward_baseline_v1,legacy_default,wf_2020,220.777205,59.775308,161.001897,54.275144,0.791023,False
1,soxlbacktest_qlib_walkforward_baseline_v1,legacy_default,wf_2022,-79.396371,-86.588072,7.191701,84.918053,0.866930,False
2,soxlbacktest_qlib_walkforward_baseline_v1,legacy_default,wf_2023_2024,2.935118,-2.603429,5.538546,32.461810,0.310435,False
3,soxlbacktest_qlib_walkforward_baseline_v1,wf_pass_ytd18_15_15_02,wf_2020,29.156603,59.775308,-30.618705,12.785207,0.171011,False
4,soxlbacktest_qlib_walkforward_baseline_v1,wf_pass_ytd18_15_15_02,wf_2022,-16.044551,-86.588072,70.543521,18.514585,0.102318,True
5,soxlbacktest_qlib_walkforward_baseline_v1,wf_pass_ytd18_15_15_02,wf_2023_2024,4.505779,-2.603429,7.109208,1.061804,0.013221,False
6,soxlbacktest_qlib_walkforward_baseline_v1,low_risk_15_05_02,wf_2020,27.643507,59.775308,-32.131801,6.828971,0.134726,False
7,soxlbacktest_qlib_walkforward_baseline_v1,low_risk_15_05_02,wf_2022,-16.864520,-86.588072,69.723552,19.456050,0.108641,True
8,soxlbacktest_qlib_walkforward_baseline_v1,low_risk_15_05_02,wf_2023_2024,1.271410,-2.603429,3.874838,1.217418,0.013428,False
9,cap10_three_fold_checkpoint_metric_replay_v2,bc_initialization,wf_2020,NaN,NaN,-292.753021,14.984701,NaN,False


In [3]:

def window_index(start: str, end: str) -> pd.DatetimeIndex:
    return panel.loc[start:end].index


def backtest_position_path(position: pd.Series, start: str, end: str) -> tuple[pd.DataFrame, dict[str, Any]]:
    idx = window_index(start, end)
    px = price.reindex(idx).astype(float)
    daily_ret = px.pct_change().fillna(0.0)
    pos = position.reindex(idx).ffill().fillna(0.0).clip(0.0, 1.0)
    strat_nav = (1.0 + pos.shift(1).fillna(0.0) * daily_ret).cumprod()
    buyhold_nav = px / float(px.iloc[0])
    drawdown = strat_nav / strat_nav.cummax() - 1.0
    daily = pd.DataFrame({
        'date': idx,
        'price': px.to_numpy(dtype=float),
        'position': pos.to_numpy(dtype=float),
        'strategy_nav': strat_nav.to_numpy(dtype=float) * 100000.0,
        'buyhold_nav': buyhold_nav.to_numpy(dtype=float) * 100000.0,
        'drawdown': drawdown.to_numpy(dtype=float),
    })
    metrics = {
        'start': idx[0].strftime('%Y-%m-%d'),
        'end': idx[-1].strftime('%Y-%m-%d'),
        'rows': int(len(idx)),
        'return': float((strat_nav.iloc[-1] - 1.0) * 100.0),
        'buyhold_return': float((buyhold_nav.iloc[-1] - 1.0) * 100.0),
        'relative_return': float((strat_nav.iloc[-1] - buyhold_nav.iloc[-1]) * 100.0),
        'max_dd': float(abs(drawdown.min()) * 100.0),
        'avg_pos': float(pos.mean()),
        'end_pos': float(pos.iloc[-1]),
        'trades': int((pos.diff().abs() > 0.01).sum()),
    }
    metrics['gate'] = bool(metrics['relative_return'] >= TARGET_RELATIVE_RETURN and metrics['max_dd'] < TARGET_MAX_DD)
    return daily, metrics


def score_metrics(metrics_by_window: dict[str, dict[str, Any]]) -> tuple[bool, float]:
    gate = True
    score = 0.0
    for metrics in metrics_by_window.values():
        gate = gate and bool(metrics['gate'])
        score += max(0.0, TARGET_RELATIVE_RETURN - float(metrics['relative_return']))
        score += max(0.0, float(metrics['max_dd']) - TARGET_MAX_DD) * 5.0
    return bool(gate), float(score)


def flatten_metrics(candidate: str, family: str, params: dict[str, Any], position: pd.Series) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    row = {'candidate': candidate, 'family': family, **params}
    rows = []
    metrics_by_window = {}
    for window, (start, end) in WINDOWS.items():
        daily, metrics = backtest_position_path(position, start, end)
        metrics_by_window[window] = metrics
        rows.append({'candidate': candidate, 'family': family, 'window': window, **metrics})
        for key, value in metrics.items():
            if key in {'start', 'end', 'rows'}:
                continue
            row[f'{window}_{key}'] = value
    all_gate, score = score_metrics(metrics_by_window)
    row['all_gate'] = all_gate
    row['score'] = score
    row['gate_count'] = int(sum(bool(item['gate']) for item in metrics_by_window.values()))
    row['min_relative_return'] = min(float(item['relative_return']) for item in metrics_by_window.values())
    row['max_max_dd'] = max(float(item['max_dd']) for item in metrics_by_window.values())
    row['mean_avg_pos'] = float(np.mean([item['avg_pos'] for item in metrics_by_window.values()]))
    return row, rows

# Legacy teacher baseline under the notebook's metric function.
legacy_summary, legacy_window_rows = flatten_metrics('legacy_teacher', 'baseline', {}, legacy_position)
legacy_context = pd.DataFrame(legacy_window_rows)
legacy_context.to_csv(CARD / 'legacy_teacher_window_metrics.csv', index=False)
display(legacy_context)


,candidate,family,window,start,end,rows,return,buyhold_return,relative_return,max_dd,avg_pos,end_pos,trades,gate
0,legacy_teacher,baseline,wf_2020,2020-01-02,2020-12-31,253,231.353241,59.775308,171.577932,54.189496,0.791023,0.9813,8,False
1,legacy_teacher,baseline,wf_2022,2022-01-03,2022-12-30,251,-79.349158,-86.588072,7.238914,84.800431,0.866930,0.9087,11,False
2,legacy_teacher,baseline,wf_2023_2024,2024-01-02,2024-12-31,252,5.434633,-2.603429,8.038062,32.166358,0.310435,0.6871,12,False


In [4]:

def feature_cap_position(params: dict[str, Any]) -> tuple[pd.Series, dict[str, int]]:
    vq = float(params['vixq'])
    rv = float(params['realvol'])
    neg63 = float(params['neg63'])
    neg180 = float(params['neg180'])
    dd_thr = float(params['dd_thr'])
    risk_cap = float(params['risk_cap'])
    stress_cap = float(params['stress_cap'])
    risk = (
        ((feature_frame['vixq_252'] > vq) & (feature_frame['realvol_21'] > rv))
        | ((feature_frame['trend63'] < neg63) & (feature_frame['ret21'] < 0.0))
        | ((feature_frame['trend180'] < neg180) & (feature_frame['ret63'] < 0.0))
        | (feature_frame['dd63'] < dd_thr)
    ).fillna(False)
    stress = (
        ((feature_frame['vixq_252'] > max(vq, 0.93)) & (feature_frame['realvol_21'] > max(rv, 1.1)))
        | ((feature_frame['trend63'] < neg63) & (feature_frame['trend180'] < neg180) & (feature_frame['ret21'] < 0.0))
    ).fillna(False)
    pos = legacy_position.copy()
    pos.loc[risk] = np.minimum(pos.loc[risk], risk_cap)
    pos.loc[stress] = np.minimum(pos.loc[stress], stress_cap)
    return pos.clip(0.0, 1.0), {'risk_days': int(risk.sum()), 'stress_days': int(stress.sum())}

feature_rows = []
feature_window_rows = []
feature_grid = itertools.product(
    [0.85, 0.90, 0.95],
    [0.90, 1.10, 1.30],
    [-0.03, -0.06],
    [-0.02, -0.05],
    [-0.15, -0.22],
    [0.05, 0.10, 0.15, 0.20, 0.30, 0.40],
    [0.00, 0.05, 0.10],
)
for idx, (vixq, realvol, neg63, neg180, dd_thr, risk_cap, stress_cap) in enumerate(feature_grid):
    params = {
        'vixq': vixq,
        'realvol': realvol,
        'neg63': neg63,
        'neg180': neg180,
        'dd_thr': dd_thr,
        'risk_cap': risk_cap,
        'stress_cap': stress_cap,
    }
    pos, counts = feature_cap_position(params)
    row, rows = flatten_metrics(f'feature_cap_{idx:04d}', 'feature_cap', {**params, **counts}, pos)
    feature_rows.append(row)
    feature_window_rows.extend(rows)
feature_screen = pd.DataFrame(feature_rows).sort_values(['all_gate', 'score'], ascending=[False, True]).reset_index(drop=True)
feature_window_metrics = pd.DataFrame(feature_window_rows)
feature_screen.to_csv(CARD / 'feature_cap_screen.csv', index=False)
feature_window_metrics.to_csv(CARD / 'feature_cap_window_metrics.csv', index=False)
display(feature_screen.head(20))


,candidate,family,vixq,realvol,neg63,neg180,dd_thr,risk_cap,stress_cap,risk_days,...,wf_2023_2024_avg_pos,wf_2023_2024_end_pos,wf_2023_2024_trades,wf_2023_2024_gate,all_gate,score,gate_count,min_relative_return,max_max_dd,mean_avg_pos
0,feature_cap_1230,feature_cap,0.95,1.3,-0.06,-0.02,-0.15,0.15,0.00,1905,...,0.103988,0.15,37,False,False,150.122121,0,1.762888,38.816343,0.223048
1,feature_cap_0942,feature_cap,0.95,0.9,-0.06,-0.02,-0.15,0.15,0.00,1909,...,0.103988,0.15,37,False,False,150.462291,0,1.762888,38.816343,0.222653
2,feature_cap_1086,feature_cap,0.95,1.1,-0.06,-0.02,-0.15,0.15,0.00,1906,...,0.103988,0.15,37,False,False,150.462291,0,1.762888,38.816343,0.222653
3,feature_cap_1158,feature_cap,0.95,1.3,-0.03,-0.02,-0.15,0.15,0.00,1910,...,0.103393,0.15,39,False,False,150.549699,0,1.335310,38.816343,0.222850
4,feature_cap_0870,feature_cap,0.95,0.9,-0.03,-0.02,-0.15,0.15,0.00,1914,...,0.103393,0.15,39,False,False,150.889869,0,1.335310,38.816343,0.222455
5,feature_cap_1014,feature_cap,0.95,1.1,-0.03,-0.02,-0.15,0.15,0.00,1911,...,0.103393,0.15,39,False,False,150.889869,0,1.335310,38.816343,0.222455
6,feature_cap_1231,feature_cap,0.95,1.3,-0.06,-0.02,-0.15,0.15,0.05,1905,...,0.116092,0.15,37,False,False,150.955179,0,2.502797,39.130937,0.239948
7,feature_cap_0366,feature_cap,0.85,1.3,-0.06,-0.02,-0.15,0.15,0.00,1905,...,0.103393,0.15,35,False,False,150.957273,0,0.927737,38.816343,0.222850
8,feature_cap_0798,feature_cap,0.90,1.3,-0.06,-0.02,-0.15,0.15,0.00,1905,...,0.103393,0.15,35,False,False,150.957273,0,0.927737,38.816343,0.222850
9,feature_cap_1159,feature_cap,0.95,1.3,-0.03,-0.02,-0.15,0.15,0.05,1910,...,0.115695,0.15,39,False,False,151.242358,0,2.215618,39.130937,0.239816


In [5]:

def equity_budget_position(raw_position: pd.Series, params: dict[str, Any], start: str, end: str) -> pd.Series:
    start_dd = float(params['start_dd'])
    mid_dd = float(params['mid_dd'])
    hard_dd = float(params['hard_dd'])
    mid_cap = float(params['mid_cap'])
    hard_cap = float(params['hard_cap'])
    idx = window_index(start, end)
    px = price.reindex(idx).astype(float)
    daily_ret = px.pct_change().fillna(0.0)
    raw = raw_position.reindex(idx).ffill().fillna(0.0).clip(0.0, 1.0)
    nav = 1.0
    peak = 1.0
    prev_pos = 0.0
    out = []
    for dt in idx:
        nav *= 1.0 + prev_pos * float(daily_ret.loc[dt])
        peak = max(peak, nav)
        current_dd = 1.0 - nav / peak if peak else 0.0
        target = float(raw.loc[dt])
        if current_dd >= hard_dd:
            target = min(target, hard_cap)
        elif current_dd >= mid_dd:
            target = min(target, mid_cap)
        elif current_dd >= start_dd:
            target = min(target, max(mid_cap, (mid_cap + target) / 2.0))
        prev_pos = float(np.clip(target, 0.0, 1.0))
        out.append(prev_pos)
    return pd.Series(out, index=idx)


def full_equity_budget_position(raw_position: pd.Series, params: dict[str, Any]) -> pd.Series:
    parts = []
    for start, end in WINDOWS.values():
        parts.append(equity_budget_position(raw_position, params, start, end))
    return pd.concat(parts).sort_index()

budget_rows = []
budget_window_rows = []
budget_param_grid = []
for start_dd, mid_dd, hard_dd, mid_cap, hard_cap in itertools.product(
    [0.04, 0.06, 0.08, 0.10],
    [0.10, 0.12, 0.15],
    [0.16, 0.18, 0.20],
    [0.05, 0.10, 0.15, 0.20, 0.30],
    [0.00, 0.03, 0.05, 0.10],
):
    if not (start_dd < mid_dd < hard_dd):
        continue
    budget_param_grid.append({'start_dd': start_dd, 'mid_dd': mid_dd, 'hard_dd': hard_dd, 'mid_cap': mid_cap, 'hard_cap': hard_cap})

for idx, params in enumerate(budget_param_grid):
    pos = full_equity_budget_position(legacy_position, params)
    row, rows = flatten_metrics(f'equity_budget_{idx:04d}', 'equity_budget', params, pos)
    budget_rows.append(row)
    budget_window_rows.extend(rows)
budget_screen = pd.DataFrame(budget_rows).sort_values(['all_gate', 'score'], ascending=[False, True]).reset_index(drop=True)
budget_window_metrics = pd.DataFrame(budget_window_rows)
budget_screen.to_csv(CARD / 'equity_budget_screen.csv', index=False)
budget_window_metrics.to_csv(CARD / 'equity_budget_window_metrics.csv', index=False)
display(budget_screen.head(20))


,candidate,family,start_dd,mid_dd,hard_dd,mid_cap,hard_cap,wf_2020_return,wf_2020_buyhold_return,wf_2020_relative_return,...,wf_2023_2024_avg_pos,wf_2023_2024_end_pos,wf_2023_2024_trades,wf_2023_2024_gate,all_gate,score,gate_count,min_relative_return,max_max_dd,mean_avg_pos
0,equity_budget_0404,equity_budget,0.08,0.10,0.20,0.10,0.00,5.060870,59.775308,-54.714438,...,0.137460,0.10,5,False,False,95.433723,0,-54.714438,20.086365,0.115222
1,equity_budget_0224,equity_budget,0.06,0.10,0.20,0.10,0.00,6.022130,59.775308,-53.753178,...,0.133587,0.10,6,False,False,99.839311,0,-53.753178,20.857803,0.118949
2,equity_budget_0580,equity_budget,0.10,0.12,0.20,0.05,0.00,0.156033,59.775308,-59.619275,...,0.115736,0.05,6,False,False,100.257891,0,-59.619275,20.110955,0.101941
3,equity_budget_0561,equity_budget,0.10,0.12,0.18,0.05,0.03,0.141110,59.775308,-59.634199,...,0.115736,0.05,6,False,False,101.925621,0,-59.634199,20.441517,0.103107
4,equity_budget_0581,equity_budget,0.10,0.12,0.20,0.05,0.03,0.156033,59.775308,-59.619275,...,0.115736,0.05,6,False,False,102.176335,0,-59.619275,20.494644,0.105500
5,equity_budget_0542,equity_budget,0.10,0.12,0.16,0.05,0.05,0.156033,59.775308,-59.619275,...,0.115736,0.05,6,False,False,103.456987,0,-59.619275,20.750774,0.105792
6,equity_budget_0562,equity_budget,0.10,0.12,0.18,0.05,0.05,0.156033,59.775308,-59.619275,...,0.115736,0.05,6,False,False,103.456987,0,-59.619275,20.750774,0.105792
7,equity_budget_0582,equity_budget,0.10,0.12,0.20,0.05,0.05,0.156033,59.775308,-59.619275,...,0.115736,0.05,6,False,False,103.456987,0,-59.619275,20.750774,0.105792
8,equity_budget_0205,equity_budget,0.06,0.10,0.18,0.10,0.03,0.252851,59.775308,-59.522457,...,0.133587,0.10,6,False,False,103.604909,0,-59.522457,20.457067,0.118356
9,equity_budget_0385,equity_budget,0.08,0.10,0.18,0.10,0.03,3.066163,59.775308,-56.709146,...,0.137460,0.10,5,False,False,103.624091,0,-56.709146,21.325497,0.119028


In [6]:

# Combined screen: only combine the strongest feature-cap candidates with all budget candidates.
combined_rows = []
combined_window_rows = []
top_feature_candidates = feature_screen.head(20).copy()
for fidx, feature_row in top_feature_candidates.iterrows():
    feature_params = {key: feature_row[key] for key in ['vixq', 'realvol', 'neg63', 'neg180', 'dd_thr', 'risk_cap', 'stress_cap']}
    feature_pos, feature_counts = feature_cap_position(feature_params)
    for bidx, budget_params in enumerate(budget_param_grid):
        pos = full_equity_budget_position(feature_pos, budget_params)
        candidate = f'combined_f{int(fidx):02d}_b{bidx:03d}'
        params = {
            **{f'feature_{key}': value for key, value in feature_params.items()},
            **{f'feature_{key}': value for key, value in feature_counts.items()},
            **budget_params,
        }
        row, rows = flatten_metrics(candidate, 'feature_cap_plus_equity_budget', params, pos)
        combined_rows.append(row)
        combined_window_rows.extend(rows)
combined_screen = pd.DataFrame(combined_rows).sort_values(['all_gate', 'score'], ascending=[False, True]).reset_index(drop=True)
combined_window_metrics = pd.DataFrame(combined_window_rows)
combined_screen.to_csv(CARD / 'combined_screen.csv', index=False)
combined_window_metrics.to_csv(CARD / 'combined_window_metrics.csv', index=False)
display(combined_screen.head(20))


,candidate,family,feature_vixq,feature_realvol,feature_neg63,feature_neg180,feature_dd_thr,feature_risk_cap,feature_stress_cap,feature_risk_days,...,wf_2023_2024_avg_pos,wf_2023_2024_end_pos,wf_2023_2024_trades,wf_2023_2024_gate,all_gate,score,gate_count,min_relative_return,max_max_dd,mean_avg_pos
0,combined_f00_b416,feature_cap_plus_equity_budget,0.95,1.3,-0.06,-0.02,-0.15,0.15,0.00,1905,...,0.102883,0.15,38,False,False,88.903116,0,-28.346047,20.69291,0.140290
1,combined_f03_b416,feature_cap_plus_equity_budget,0.95,1.3,-0.03,-0.02,-0.15,0.15,0.00,1910,...,0.102288,0.15,40,False,False,89.335629,0,-28.346047,20.69291,0.140092
2,combined_f06_b416,feature_cap_plus_equity_budget,0.95,1.3,-0.06,-0.02,-0.15,0.15,0.05,1905,...,0.114987,0.15,38,False,False,89.440484,0,-28.618416,20.89560,0.151944
3,combined_f09_b416,feature_cap_plus_equity_budget,0.95,1.3,-0.03,-0.02,-0.15,0.15,0.05,1910,...,0.114590,0.15,40,False,False,89.730978,0,-28.618416,20.89560,0.151812
4,combined_f01_b416,feature_cap_plus_equity_budget,0.95,0.9,-0.06,-0.02,-0.15,0.15,0.00,1909,...,0.102883,0.15,38,False,False,89.743685,0,-29.186617,20.69291,0.139895
5,combined_f02_b416,feature_cap_plus_equity_budget,0.95,1.1,-0.06,-0.02,-0.15,0.15,0.00,1906,...,0.102883,0.15,38,False,False,89.743685,0,-29.186617,20.69291,0.139895
6,combined_f07_b416,feature_cap_plus_equity_budget,0.85,1.3,-0.06,-0.02,-0.15,0.15,0.00,1905,...,0.102288,0.15,36,False,False,89.747907,0,-28.346047,20.69291,0.140092
7,combined_f08_b416,feature_cap_plus_equity_budget,0.90,1.3,-0.06,-0.02,-0.15,0.15,0.00,1905,...,0.102288,0.15,36,False,False,89.747907,0,-28.346047,20.69291,0.140092
8,combined_f14_b416,feature_cap_plus_equity_budget,0.95,0.9,-0.06,-0.02,-0.15,0.15,0.05,1909,...,0.114987,0.15,38,False,False,89.990046,0,-29.167979,20.89560,0.151681
9,combined_f15_b416,feature_cap_plus_equity_budget,0.95,1.1,-0.06,-0.02,-0.15,0.15,0.05,1906,...,0.114987,0.15,38,False,False,89.990046,0,-29.167979,20.89560,0.151681


In [7]:

all_screens = pd.concat([
    feature_screen.assign(screen='feature_cap'),
    budget_screen.assign(screen='equity_budget'),
    combined_screen.assign(screen='combined'),
], ignore_index=True, sort=False)
all_screens = all_screens.sort_values(['all_gate', 'score'], ascending=[False, True]).reset_index(drop=True)
all_screens.to_csv(CARD / 'candidate_screen_summary.csv', index=False)
selected = all_screens.iloc[0].to_dict()
selected_candidate = str(selected['candidate'])
selected_family = str(selected['family'])
selected_windows = []
for window in WINDOWS:
    selected_windows.append({
        'window': window,
        'return': selected.get(f'{window}_return'),
        'buyhold_return': selected.get(f'{window}_buyhold_return'),
        'relative_return': selected.get(f'{window}_relative_return'),
        'max_dd': selected.get(f'{window}_max_dd'),
        'avg_pos': selected.get(f'{window}_avg_pos'),
        'gate': selected.get(f'{window}_gate'),
    })
selected_window_metrics = pd.DataFrame(selected_windows)
selected_window_metrics.to_csv(CARD / 'selected_window_metrics.csv', index=False)

all_gate_count = int(all_screens['all_gate'].sum())
if all_gate_count > 0:
    decision = 'teacher_risk_budget_feasible_ready_for_foldlocal_target_training'
    next_mechanism = 'convert_selected_teacher_budget_recipe_to_foldlocal_targets_then_run_bc_ppo_retention_candidate'
else:
    decision = 'teacher_risk_budget_not_feasible_no_promotion'
    next_mechanism = 'train_state_conditioned_multi_objective_labels_rather_than_more_soxlbacktest_teacher_caps'

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': [
        'soxlbacktest_qlib_walkforward_baseline_v1',
        'cap10_three_fold_checkpoint_metric_replay_v2',
        'cap10_training_objective_handoff_v1',
    ],
    'reference_baseline': 'soxlbacktest_qlib_buysell_reference',
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'training_target', 'execution_budget_feasibility'],
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'required_windows': list(WINDOWS),
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': False,
    'all_gate_count': all_gate_count,
    'candidate_count': int(len(all_screens)),
    'selected_candidate': selected_candidate,
    'selected_family': selected_family,
    'selected_score': float(selected['score']),
    'selected_gate_count': int(selected['gate_count']),
    'selected_min_relative_return': float(selected['min_relative_return']),
    'selected_max_max_dd': float(selected['max_max_dd']),
    'selected_mean_avg_pos': float(selected['mean_avg_pos']),
    'selected_windows': selected_windows,
    'historical_context': historical_context,
    'promotion_blocker': 'Diagnostic teacher-risk-budget screen only; no actor was trained or retained. Full completion still requires fixed-split plus wf_2020/wf_2022/wf_2023_2024 retained actor replay and YTD alignment to existing best.',
    'sidecars': [
        'historical_context.json',
        'historical_gate_context.csv',
        'legacy_teacher_window_metrics.csv',
        'feature_cap_screen.csv',
        'feature_cap_window_metrics.csv',
        'equity_budget_screen.csv',
        'equity_budget_window_metrics.csv',
        'combined_screen.csv',
        'combined_window_metrics.csv',
        'candidate_screen_summary.csv',
        'selected_window_metrics.csv',
        'selection.json',
        'summary.json',
        'conclusion.md',
    ],
}
summary = {
    **selection,
    'conclusion': [
        f"Decision: {decision}.",
        f"Selected {selected_candidate} from {selected_family}; gate_count={selection['selected_gate_count']}/3, min_relative={selection['selected_min_relative_return']:.2f}pp, max_dd={selection['selected_max_max_dd']:.2f}%.",
        'SOXLBacktest teacher caps are diagnostic only and are not promoted as runtime overlays or retained actors.',
        f"Next mechanism: {next_mechanism}.",
    ],
}
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
conclusion_lines = [
    '# Conclusion',
    '',
    f"Decision: `{decision}`.",
    '',
    f"Selected candidate: `{selected_candidate}` from `{selected_family}`.",
    f"Gate count: {selection['selected_gate_count']}/3.",
    f"Minimum relative return: {selection['selected_min_relative_return']:.2f}pp.",
    f"Worst MaxDD: {selection['selected_max_max_dd']:.2f}%.",
    '',
    'Selected window metrics:',
]
def fmt_metric(value, suffix='', digits=2):
    if value is None or pd.isna(value):
        return 'NA'
    return f'{float(value):.{digits}f}{suffix}'

for item in selected_windows:
    conclusion_lines.append(
        f"- {item['window']}: rel={fmt_metric(item['relative_return'], 'pp')}, MaxDD={fmt_metric(item['max_dd'], '%')}, avg_pos={fmt_metric(item['avg_pos'], '', 3)}, gate={bool(item['gate'])}"
    )
conclusion_lines.extend([
    '',
    'Risk note: no actor bundle is retained and no runtime overlay is promoted from this screen.',
    f"Next action: `{next_mechanism}`.",
])
(CARD / 'conclusion.md').write_text('\n'.join(conclusion_lines) + '\n', encoding='utf-8')
print(json.dumps(json_safe(selection), indent=2, allow_nan=False))
display(selected_window_metrics)


{
  "experiment": "soxlbacktest_teacher_risk_budget_feasibility_v1",
  "created_at": "2026-06-13T13:02:49.311716+00:00",
  "decision": "teacher_risk_budget_not_feasible_no_promotion",
  "next_mechanism": "train_state_conditioned_multi_objective_labels_rather_than_more_soxlbacktest_teacher_caps",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": [
    "soxlbacktest_qlib_walkforward_baseline_v1",
    "cap10_three_fold_checkpoint_metric_replay_v2",
    "cap10_training_objective_handoff_v1"
  ],
  "reference_baseline": "soxlbacktest_qlib_buysell_reference",
  "role": "diagnose",
  "changed_layer": [
    "model_selection",
    "training_target",
    "execution_budget_feasibility"
  ],
  "target_relative_return_pp": 30.0,
  "target_max_dd_pct": 20.0,
  "required_windows": [
    "wf_2020",
    "wf_2022",
    "wf_2023_2024"
  ],
  "promotion_allowed": false,
  "actor_retention_allowed": false,
  "training_allowed_from_this_card": false,
  "all_gat

,window,return,buyhold_return,relative_return,max_dd,avg_pos,gate
0,wf_2020,31.429262,59.775308,-28.346047,18.990948,0.294326,False
1,wf_2022,-20.124977,-86.588072,66.463094,20.692910,0.023662,False
2,wf_2023_2024,0.304054,-2.603429,2.907483,18.087109,0.102883,False


In [8]:

md = '\n'.join([
    '## Conclusion',
    '',
    *summary['conclusion'],
    '',
    'Artifacts: `candidate_screen_summary.csv`, `selected_window_metrics.csv`, `selection.json`, `summary.json`, `conclusion.md`.',
])
display(Markdown(md))


## Conclusion

Decision: teacher_risk_budget_not_feasible_no_promotion.
Selected combined_f00_b416 from feature_cap_plus_equity_budget; gate_count=0/3, min_relative=-28.35pp, max_dd=20.69%.
SOXLBacktest teacher caps are diagnostic only and are not promoted as runtime overlays or retained actors.
Next mechanism: train_state_conditioned_multi_objective_labels_rather_than_more_soxlbacktest_teacher_caps.

Artifacts: `candidate_screen_summary.csv`, `selected_window_metrics.csv`, `selection.json`, `summary.json`, `conclusion.md`.